## Disparo automático do envio de email (GitHub Actions)

Esta célula conecta o pipeline do Databricks ao GitHub Actions, automatizando o envio
do relatório por email.

**Por que isso é necessário:** o Databricks Free Edition bloqueia o envio direto de
email (restrição de rede de saída). A solução é delegar o envio ao GitHub Actions —
e esta célula é o gatilho que aciona esse processo.

**O que ela faz:**
1. Lê o token de acesso do GitHub a partir de um arquivo no Volume (mantido fora do
   código por segurança)
2. Chama a API do GitHub solicitando a execução do workflow de envio de email
3. Confirma o disparo (status HTTP 204 indica sucesso)

Ao rodar com sucesso, o GitHub Actions é acionado: ele baixa o relatório gerado
(via Databricks Files API) e o envia por email automaticamente.

**Fluxo completo:** pipeline gera o relatório → esta célula dispara o Actions →
Actions baixa o relatório e envia o email.

In [0]:
import urllib.request
import json

# lê o token do GitHub do arquivo no Volume (não hardcoded)
with open("/Volumes/workspace/default/tokens/git_token.txt") as f:
    github_token = f.read().strip()

# configuração do repositório e workflow
OWNER = "jotajoaolucas"
REPO = "review-insights-llm-databricks"
WORKFLOW = "enviar_relatorio_auto.yml"
BRANCH = "main"

url = f"https://api.github.com/repos/{OWNER}/{REPO}/actions/workflows/{WORKFLOW}/dispatches"

payload = json.dumps({"ref": BRANCH}).encode("utf-8")

req = urllib.request.Request(
    url,
    data=payload,
    method="POST",
    headers={
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json",
        "User-Agent": "databricks-pipeline",
        "Content-Type": "application/json",
    }
)

try:
    resposta = urllib.request.urlopen(req, timeout=15)
    if resposta.status == 204:
        print("✅ GitHub Actions disparado com sucesso!")
        print("O email será enviado em instantes.")
    else:
        print(f"Resposta inesperada: {resposta.status}")
except Exception as e:
    print("❌ Erro ao disparar o workflow:", e)